# 01 · 什麼是 Agent？載入本地 Qwen

歡迎來到 **AI Agent → 手刻 AI Agent**。

上一條軌道你**從零刻了一個迷你 GPT**，徹底懂了 LLM 內部。但那隻太小、做不了事。這條軌道換目標：**讓模型真的去做事情**——所以換一個「真正能用」的模型，並在它外面**親手刻一圈控制邏輯**。那圈邏輯就是 Agent。

> **Agent = LLM（推理引擎）+ 工具（tools）+ 迴圈（loop）。**

> 💡 Colab 記得開 **T4 GPU**：執行階段 → 變更執行階段類型 → T4 GPU。本軌道零框架，每一圈迴圈都你自己寫。

## 1. 安裝相依套件

第一次跑約 1–2 分鐘。

In [ ]:
%pip install -q -U "transformers>=4.45" accelerate bitsandbytes

## 2. 用 4-bit 量化載入 Qwen

`bitsandbytes` 把模型砍成 4-bit，1.5B 量化後只吃 1–2GB 顯示記憶體，免費 T4 綽綽有餘。整條軌道只透過 `chat()` 這一個函式碰模型。

In [ ]:
import json, re
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"   # 想更強可換 Qwen2.5-3B-Instruct，T4 也裝得下

_bnb = BitsAndBytesConfig(
    load_in_4bit=True,                    # 4-bit 量化：模型體積砍約 75%
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, quantization_config=_bnb, device_map="auto", torch_dtype=torch.float16,
)
model.eval()
print(f"已載入 {MODEL_ID}（4-bit）於 {model.device}")


@torch.no_grad()
def chat(messages, max_new_tokens=512, temperature=0.0):
    """LLM 的唯一介面：丟一串 messages，回模型吐的純文字。

    messages = [{"role": "system"|"user"|"assistant", "content": "..."}]
    temperature=0 → 貪婪解碼（穩定、可重現），>0 → 取樣（多樣）。
    整條軌道只透過這個函式碰模型；要換成別的模型/API，改這裡就好。
    """
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    gen = dict(max_new_tokens=max_new_tokens, pad_token_id=tokenizer.eos_token_id)
    if temperature > 0:
        gen.update(do_sample=True, temperature=temperature, top_p=0.9)
    else:
        gen.update(do_sample=False)
    out = model.generate(**inputs, **gen)
    reply = out[0][inputs.input_ids.shape[1]:]
    return tokenizer.decode(reply, skip_special_tokens=True).strip()

## 3. 跟模型對話

LLM 只會做一件事：給它一串訊息，它接著吐一串文字。

In [ ]:
reply = chat([{"role": "user", "content": "用一句話解釋什麼是大型語言模型。"}])
print(reply)

### 它背過的東西，答得出來

In [ ]:
print(chat([{"role": "user", "content": "2 的 10 次方是多少？"}]))

### 但它碰不到的世界，只能瞎掰

模型沒有時鐘——它的世界停在訓練資料那一刻。問它現在幾點，它只會編一個。

In [ ]:
print(chat([{"role": "user", "content": "現在台北時間幾點幾分？請只回答時間。"}]))
# ☝️ 這個時間是模型瞎掰的，不是真的。這個「答不出來」正是整條軌道的起點。

## 小結

- **Agent = LLM + 工具 + 迴圈**。LLM 只會「文字進、文字出」。
- 我們用 **4-bit 量化**在免費 T4 上載入了 Qwen，封裝成 `chat()`。
- 模型**背過的**答得出來，**碰不到的真實世界**（時間、計算、查資料）只能瞎掰。

下一課：教模型在卡住時，改成**呼叫一個工具**。